In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# 1. Load the data again just to keep this notebook self-contained
file_path = '../data/alzheimers_prediction_dataset.csv'
df = pd.read_csv(file_path)

# 2. Keep ONLY the 13 features from our React form (plus the target)
columns_to_keep = [
    "Age", "Education Level", "BMI", "Cognitive Test Score", 
    "Physical Activity Level", "Smoking Status", "Alcohol Consumption", 
    "Diabetes", "Hypertension", "Cholesterol Level", 
    "Family History of Alzheimer’s", "Depression Level", "Sleep Quality",
    "Alzheimer’s Diagnosis"
]
df_clean = df[columns_to_keep]

# 3. Separate Features (X) and the Target Variable (y)
X = df_clean.drop(columns=['Alzheimer’s Diagnosis'])
y = df_clean['Alzheimer’s Diagnosis']

# 4. Encode the target variable (Yes = 1, No = 0)
y = y.map({'Yes': 1, 'No': 0})

# 5. One-Hot Encode all the categorical text columns
X_encoded = pd.get_dummies(X, drop_first=True)

# 6. Scale the features (Crucial for SVM and Logistic Regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_encoded.columns)

# 7. Apply SMOTE to balance the classes perfectly
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_scaled_df, y)

# 8. Safely print the results to verify
print("Original target distribution:")
print(y.value_counts())

print("\nBalanced target distribution after SMOTE:")
print(y_balanced.value_counts())

print("\nNew Feature Shape:")
print(X_balanced.shape)

Original target distribution:
Alzheimer’s Diagnosis
0    43570
1    30713
Name: count, dtype: int64

Balanced target distribution after SMOTE:
Alzheimer’s Diagnosis
0    43570
1    43570
Name: count, dtype: int64

New Feature Shape:
(87140, 18)


In [4]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import numpy as np

# 9. Apply PCA to reduce dimensionality (keeping 95% of the variance)
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_balanced)

print(f"Original number of features: {X_balanced.shape[1]}")
print(f"Number of features after PCA: {X_pca.shape[1]}")

# 10. Split the data into 70% Training and 30% Testing
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, 
    y_balanced, 
    test_size=0.30, 
    random_state=42, 
    stratify=y_balanced # Ensures the 70/30 split keeps the perfectly balanced 50/50 yes/no ratio
)

print(f"Training Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")

# 11. Save the processed data so we can load it cleanly into our model training notebooks
np.save('../data/X_train.npy', X_train)
np.save('../data/X_test.npy', X_test)
np.save('../data/y_train.npy', y_train)
np.save('../data/y_test.npy', y_test)
print("\nData successfully split and saved! Ready for Phase 4.")

Original number of features: 18
Number of features after PCA: 17
Training Data Shape: (60998, 17)
Testing Data Shape: (26142, 17)

Data successfully split and saved! Ready for Phase 4.
